# Notebook 2 — why Monte Carlo packets solve the same equation

Level-1 code: $P(\tau_{\rm int} > \tau) = e^{-\tau}$, so $\tau_{\rm MC} = -\ln \xi$. Escape fractions at $10^2 \ldots 10^5$ packets against $e^{-\tau}$; the histogram of draws; the convergence of the error.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))   # rtedu, uninstalled (education/src)
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI, GROUP_COLOUR
rng = np.random.default_rng(rtedu.SEEDS["ch02"])

## Level 1: one line of physics

A packet is transmitted through a slab of depth $\tau$ if its interaction depth exceeds $\tau$. Nothing else happens in this chapter.

In [ ]:
def escape_fraction(rng, tau_slab, n):
    tau_int = -np.log(1.0 - rng.random(n))     # the exponential draw
    return float(np.mean(tau_int > tau_slab))

tau_slab = 1.0
Ns = [10**2, 10**3, 10**4, 10**5]
p_exact = float(np.exp(-tau_slab))
fractions = {N: escape_fraction(rng, tau_slab, N) for N in Ns}
errors = {N: abs(f - p_exact) for N, f in fractions.items()}
sigmas = {N: float(np.sqrt(p_exact * (1 - p_exact) / N)) for N in Ns}
for N in Ns:
    print(f"N = {N:>6d}: escape {fractions[N]:.4f}  exact {p_exact:.4f}  |error| {errors[N]:.4f}  binomial sigma {sigmas[N]:.4f}")

## The sampled depths are exponential

The histogram of $10^5$ draws against $e^{-\tau}$, and the Kolmogorov statistic (its 1 % critical value is $1.63/\sqrt{N}$).

In [ ]:
n = 100_000
draws = -np.log(1.0 - rng.random(n))
srt = np.sort(draws); F = -np.expm1(-srt); i = np.arange(1, n + 1)
ks_D = float(max(np.max(i / n - F), np.max(F - (i - 1) / n)))
ks_crit = 1.63 / np.sqrt(n)
print(f"KS D = {ks_D:.4f} against the 1% critical value {ks_crit:.4f}; mean {draws.mean():.4f}, var {draws.var():.4f}")
assert ks_D < ks_crit

## Validation against `rtedu`

`rtedu.packets.sample_tau` is the same draw; `propagate_slab` with no scattering is the same single-draw experiment.

In [ ]:
from rtedu.packets import sample_tau, propagate_slab
d2 = sample_tau(np.random.default_rng(rtedu.SEEDS["ch02"]), 5)
assert np.allclose(d2, -np.log(1.0 - np.random.default_rng(rtedu.SEEDS["ch02"]).random(5)))
out = propagate_slab(rng, tau_slab, 10**5)
assert abs(out["transmitted"] - p_exact) < 4 * sigmas[10**5] and out["mean_interactions_transmitted"] == 0.0
print("propagate_slab:", out)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
edges = np.linspace(0, 8, 41)
axes[0].hist(draws, bins=edges, density=True, color=OI["sky"], label=r"$10^5$ draws of $-\ln\xi$")
axes[0].plot(edges, np.exp(-edges), color=OI["black"], label=r"$e^{-\tau}$"); axes[0].set_yscale("log"); axes[0].set_xlabel(r"$\tau_{\rm int}$"); axes[0].set_ylabel("density"); axes[0].legend()
axes[1].loglog(Ns, [errors[N] for N in Ns], "o-", color=OI["blue"], label="|escape fraction - $e^{-\tau}$|")
axes[1].loglog(Ns, [sigmas[N] for N in Ns], "--", color=OI["black"], label=r"binomial $\sigma = \sqrt{p(1-p)/N}$")
axes[1].set_xlabel("packets N"); axes[1].set_ylabel("error"); axes[1].legend()
fig.suptitle(r"Monte Carlo packets are a solver for $e^{-\tau}$: the error falls as $1/\sqrt{N}$", fontsize=10); fig.tight_layout()
save_fig(fig, "ch02_mc_slab")

In [ ]:
results.record("ch02", dict(tau_slab=tau_slab, p_exact=p_exact, Ns=Ns, fractions={str(N): fractions[N] for N in Ns},
                            errors={str(N): errors[N] for N in Ns}, sigmas={str(N): sigmas[N] for N in Ns},
                            ks_D=ks_D, ks_crit=ks_crit, ks_n=n, draws_mean=float(draws.mean()), draws_var=float(draws.var())))